# HW1 - Memorization and Generalization (100 Points)

When a network reaches a high test accuracy on CIFAR-10, we usually explain this by saying that it
has learned something about the structure of natural images. In this assignment we examine that
explanation by training the same network on data whose structure has been removed, and comparing
what happens in each case.

We fix one architecture, one optimizer and one set of
hyperparameters, and we change only the training data: the labels are replaced by random ones, the
pixels are permuted, or the images are replaced by Gaussian noise. For each condition we then
compare the training curve, which tells us how well the network fits the data it was given, with
the test curve, which tells us how well it does on the original task.

The assignment is organized as follows. In Part 1 we implement the transformations that corrupt
the training set. In Part 2 we define the two architectures we will use, and in Part 3 the
training and evaluation loops. Part 4 runs the different conditions and compares them, and Part 5
studies the effect when a fraction of the labels is corrupted.

**Instructions.** Fill in every cell marked `TODO`. Do not change the given function signatures,
because later cells depend on them.

## Part 0 - Setup

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets

In [ ]:
# Helper functions

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_cifar10(n_train, n_test, root="data", seed=0):
    """Random subsets of CIFAR-10 as normalized float tensors (N,3,32,32) and int64 labels."""
    mean = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(1, 3, 1, 1)
    g = torch.Generator().manual_seed(seed)
    out = []
    for train, n in [(True, n_train), (False, n_test)]:
        ds = datasets.CIFAR10(root, train=train, download=True)
        X = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float().div_(255)
        y = torch.tensor(ds.targets)
        idx = torch.randperm(len(y), generator=g)[:n]
        out += [((X[idx] - mean) / std).contiguous(), y[idx].contiguous()]
    return out


def random_crop_flip(img, pad=4):
    """Standard CIFAR augmentation applied to one (3,32,32) image."""
    img = F.pad(img, (pad, pad, pad, pad))
    i, j = torch.randint(0, 2 * pad + 1, (2,)).tolist()
    img = img[:, i:i + 32, j:j + 32]
    return img.flip(-1) if torch.rand(()) < 0.5 else img


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def save_json(obj, path):
    Path(path).write_text(json.dumps(obj))


def load_json(path):
    return json.loads(Path(path).read_text())


def show_images(X, title="", n=8):
    """Undo normalization and display the first n images of a batch."""
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    fig, axes = plt.subplots(1, n, figsize=(1.3 * n, 1.6))
    for k, ax in enumerate(axes):
        ax.imshow((X[k].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
        ax.axis("off")
    fig.suptitle(title, y=1.05)
    plt.show()


def plot_curves(histories, key, ylabel=None, ax=None, logy=False):
    """histories: {label: history dict}. Plots history[key] against epoch."""
    ax = ax or plt.gca()
    for label, h in histories.items():
        ax.plot(h[key], label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel or key)
    if logy:
        ax.set_yscale("log")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    return ax


def print_table(headers, rows):
    widths = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    line = "  ".join("-" * w for w in widths)
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print(line)
    for r in rows:
        print("  ".join(str(c).ljust(w) for c, w in zip(r, widths)))

In [ ]:
DEVICE = get_device()
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
N_TRAIN, N_TEST, EPOCHS = 10_000, 2_000, 60
set_seed(0)
print(DEVICE)

In [ ]:
Xtr, ytr, Xte, yte = load_cifar10(N_TRAIN, N_TEST) # Sometimes it may be slow and take up to 30 minutes!
print(Xtr.shape, ytr.shape, Xte.shape, yte.shape)
print(f"pixel mean {Xtr.mean():.3f}, std {Xtr.std():.3f}")

The images are standardized per color channel using the CIFAR-10 statistics, so the training set
has approximately zero mean and unit variance. We will use this fact in Part 1, where one of the
corruptions replaces the images with Gaussian noise and needs to match the mean and variance of
the real data.

## Part 1 - Randomizing the data (35 points)

In this part we implement the transformations that the later experiments apply to the training
set. The test
images and the test labels are never corrupted, so test accuracy always measures performance on
the original task and stays comparable across conditions.

We need two kinds of corruption. The first breaks the relationship between an image and its label
while leaving the images untouched, and the second destroys the structure of the images while
leaving the labels untouched. We also need a `Dataset` that serves the resulting tensors, so that
the training loop in Part 3 can iterate over them with a `DataLoader`.

In [ ]:
class ArrayDataset(torch.utils.data.Dataset):
    """Serves (image, label) pairs from tensors, optionally augmented."""

    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment

    def __len__(self):
        # TODO
        raise NotImplementedError

    def __getitem__(self, i):
        # TODO: return the i-th (image, label). If self.augment, apply random_crop_flip
        # to the image (and only to the image) first.
        raise NotImplementedError

In [ ]:
def corrupt_labels(y, p, num_classes=10, seed=0):
    """Independently with probability p, replace a label by a uniformly random class.

    Returns a new tensor; y must not be modified. Draw the randomness from a
    torch.Generator seeded with `seed` so the corruption is reproducible.
    """
    # TODO
    raise NotImplementedError

In [ ]:
for p in (0.0, 0.5, 1.0):
    changed = (corrupt_labels(ytr, p) != ytr).float().mean()
    print(f"p={p}: {changed:.3f} of labels actually changed")

At `p=1.0`, about 90% of the labels should change. The reason is that
the replacement label is drawn uniformly from all ten classes, including the original one, so
about one replacement in ten happens to reproduce the correct label. The labels are nonetheless
independent of the images, which is what the experiment requires.

In [ ]:
def randomize_inputs(X, mode, seed=0):
    """Randomize the images of X (N,3,32,32); returns a new tensor.

    'shuffled_pixels' one random permutation of the 32*32 pixel positions, applied to every
                      image and identically to all three channels. The permutation must depend
                      only on `seed`, so that calling this on the train and the test set with the
                      same seed applies the *same* permutation. The two splits then still share a
                      common spatial arrangement, although it is no longer the arrangement of a
                      natural image.
    'random_pixels'   an independent pixel permutation for each image.
    'gaussian'        i.i.d. N(0, 1) pixels. X is standardized, so noise drawn this way has the
                      same mean and variance as the real data, and the images differ from the
                      originals only in that they carry no information about the label.
    """
    # TODO
    raise NotImplementedError

In [ ]:
show_images(Xtr, "original")
for mode in ("shuffled_pixels", "random_pixels", "gaussian"):
    show_images(randomize_inputs(Xtr[:8], mode), mode)

## Part 2 - Models (30 points)

We use two architectures throughout the assignment: a multilayer perceptron with a single hidden
layer, and a small convolutional network. Both are deliberately ordinary, since what we are
studying is not the merit of a particular architecture but how each one responds to corrupted
data.

The cell after the two definitions prints the number of parameters of each model, in total and per
training example. These counts are worth reading before running any experiment, because they tell
us that both models have more than enough capacity to store the training set outright, whether or
not they turn out to use it that way.

In [ ]:
class MLP(nn.Module):
    """Flatten, then Linear -> ReLU for each width in `hidden`, then a linear classifier."""

    def __init__(self, hidden=(512,), in_dim=3 * 32 * 32, num_classes=10):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
class SmallCNN(nn.Module):
    """For each width w in `widths`: Conv3x3(padding=1) -> [BatchNorm] -> ReLU -> MaxPool2.
    Then flatten, [Dropout], and a linear classifier.

    Use bias=False in a convolution that is followed by batch norm (the norm's shift makes it
    redundant). With three pooling stages a 32x32 input leaves a 4x4 feature map.
    """

    def __init__(self, widths=(64, 128, 256), use_bn=True, dropout=0.0, num_classes=10):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
for name, model in [("MLP 1x512", MLP()), ("MLP 3x512", MLP((512,) * 3)),
                    ("SmallCNN", SmallCNN()), ("SmallCNN w/o BN", SmallCNN(use_bn=False))]:
    print(f"{name:<16} {count_params(model):>9,d} params"
          f"  ({count_params(model) / N_TRAIN:.0f} per training example)")
assert SmallCNN()(Xtr[:2]).shape == (2, 10) and MLP()(Xtr[:2]).shape == (2, 10)

## Part 3 - The training loop (25 points)

In this part we implement the two functions: one that performs a
single pass of training over a loader, and one that evaluates a model on a loader and reports the
loss and the accuracy.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """One pass over `loader`: forward, cross-entropy loss, backward, optimizer step.

    Put the model in the mode that makes batch norm and dropout behave the way they should at
    training time.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def evaluate(model, loader, device):
    """Returns (mean loss, accuracy) over `loader`.

    Put the model in the mode that makes batch norm and dropout behave the way they should at
    test time.
    """
    # TODO
    raise NotImplementedError

The driver `run(**cfg)` takes a configuration dictionary, builds the corresponding (possibly
randomized) dataset, trains a model on it, and caches the resulting history under
`results/<run_name>.json`. Because the results are cached, re-running a cell does not repeat an
experiment that has already finished. Training accuracy is measured after each epoch in eval mode
on the un-augmented training set, which keeps it comparable across conditions.

We train with Adam and an exponential learning-rate decay of 0.98 per epoch.

In [ ]:
DEFAULTS = dict(model="cnn", p=0.0, input_mode=None, augment=False, weight_decay=0.0,
                dropout=0.0, use_bn=True, epochs=EPOCHS, lr=None, seed=0)


def run_name(cfg):
    changed = [f"{k}={cfg[k]}" for k in sorted(DEFAULTS) if k != "model" and cfg[k] != DEFAULTS[k]]
    return "_".join([cfg["model"]] + changed)


def fit(model, train_ds, eval_ds, test_ds, epochs, lr, weight_decay, batch_size=128):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.98)
    loaders = dict(train=DataLoader(eval_ds, batch_size=1000),
                   test=DataLoader(test_ds, batch_size=1000))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    hist = {k: [] for k in ("train_loss", "train_acc", "test_loss", "test_acc")}
    t0 = time.time()
    for ep in range(epochs):
        train_one_epoch(model, train_loader, opt, DEVICE)
        sched.step()
        for split, loader in loaders.items():
            loss, acc = evaluate(model, loader, DEVICE)
            hist[f"{split}_loss"].append(loss)
            hist[f"{split}_acc"].append(acc)
        if ep % 20 == 19:
            print(f"    epoch {ep + 1:3d}  train_acc {hist['train_acc'][-1]:.3f}"
                  f"  test_acc {hist['test_acc'][-1]:.3f}")
    hist["seconds"] = round(time.time() - t0, 1)
    return hist


def run(**kwargs):
    cfg = {**DEFAULTS, **kwargs}
    path = RESULTS / f"{run_name(cfg)}.json"
    if path.exists():
        return load_json(path)
    print(f"running {run_name(cfg)}")
    set_seed(cfg["seed"])
    y_train = corrupt_labels(ytr, cfg["p"], seed=cfg["seed"])
    X_train, X_test = Xtr, Xte
    if cfg["input_mode"] is not None:
        # shuffled_pixels must use one permutation for both splits; the others are independent
        test_seed = cfg["seed"] if cfg["input_mode"] == "shuffled_pixels" else cfg["seed"] + 1
        X_train = randomize_inputs(Xtr, cfg["input_mode"], seed=cfg["seed"])
        X_test = randomize_inputs(Xte, cfg["input_mode"], seed=test_seed)
    if cfg["model"] == "cnn":
        model = SmallCNN(use_bn=cfg["use_bn"], dropout=cfg["dropout"])
    else:
        model = MLP((512,))
    hist = fit(model,
               ArrayDataset(X_train, y_train, augment=cfg["augment"]),
               ArrayDataset(X_train, y_train),
               ArrayDataset(X_test, yte),
               epochs=cfg["epochs"],
               lr=cfg["lr"] or 1e-3,
               weight_decay=cfg["weight_decay"])
    hist["config"] = cfg
    save_json(hist, path)
    return hist

## Part 4 - Fitting random labels and random pixels

In this part we run the five conditions and compare them. Each of them uses the same network, the
same optimizer and the same number of epochs, and only the training data differs. The first
condition uses the original data, the second replaces every label with a random one, and the
remaining three replace the images with permuted or Gaussian versions of themselves.

We run every condition with both models, and then plot the training loss, the training accuracy
and the test accuracy next to each other, so that we can see how the ability to fit the training
set and the ability to generalize behave under the same corruption. This cell performs most of the
computation in the assignment and takes about five minutes the first time it is run; afterwards
the cached results are reused.

In [ ]:
CONDITIONS = {
    "true labels": dict(),
    "random labels": dict(p=1.0),
    "shuffled pixels": dict(input_mode="shuffled_pixels"),
    "random pixels": dict(input_mode="random_pixels"),
    "gaussian": dict(input_mode="gaussian"),
}
cnn = {k: run(model="cnn", **cfg) for k, cfg in CONDITIONS.items()}
mlp = {k: run(model="mlp", **cfg) for k, cfg in CONDITIONS.items()}

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
for row, (name, hists) in enumerate([("SmallCNN", cnn), ("MLP 1x512", mlp)]):
    plot_curves(hists, "train_loss", "training loss", axes[row, 0])
    plot_curves(hists, "train_acc", "training accuracy", axes[row, 1])
    plot_curves(hists, "test_acc", "test accuracy", axes[row, 2])
    axes[row, 0].set_ylabel(f"{name}\ntraining loss")
    axes[row, 2].axhline(0.1, ls=":", c="k")
fig.tight_layout()

In [ ]:
print_table(
    ["condition", "CNN train", "CNN test", "MLP train", "MLP test"],
    [[k, f"{cnn[k]['train_acc'][-1]:.3f}", f"{cnn[k]['test_acc'][-1]:.3f}",
      f"{mlp[k]['train_acc'][-1]:.3f}", f"{mlp[k]['test_acc'][-1]:.3f}"] for k in CONDITIONS])

## Part 5 - Interpolating between signal and noise (10 points)

Part 4 compared two extremes: labels that are all correct, and labels that are all random. In this
part we examine the range in between by sweeping the corruption probability `p` from 0 to 1 and
training one model at each value.

For each run we record two quantities: the number of epochs the model needs before it fits the
training set, and the test accuracy it reaches. Plotting both against `p` lets us see how the cost
of fitting the data and the quality of the resulting model change as a growing fraction of the
training labels becomes noise. The runs at `p=0.0` and `p=1.0` are already cached from Part 4, so
only the four intermediate values are trained here.

In [ ]:
PS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
sweep = {p: run(model="cnn", p=p) for p in PS}

In [ ]:
def epochs_to_fit(history, threshold=0.99):
    """First epoch (counting from 1) at which training accuracy reaches `threshold`,
    or float("nan") if it never does."""
    # TODO
    raise NotImplementedError

In [ ]:
base = epochs_to_fit(sweep[0.0])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].plot(PS, [epochs_to_fit(sweep[p]) / base for p in PS], "o-")
axes[0].set_xlabel("label corruption p"); axes[0].set_ylabel("time to overfit (relative)")
axes[1].plot(PS, [1 - sweep[p]["test_acc"][-1] for p in PS], "o-")
axes[1].axhline(0.9, ls=":", c="k")
axes[1].set_xlabel("label corruption p"); axes[1].set_ylabel("test error")
plot_curves({f"p={p}": sweep[p] for p in PS}, "train_acc", "training accuracy", axes[2])
for ax in axes[:2]:
    ax.grid(alpha=0.3)
fig.tight_layout()

print_table(["p", "epochs to fit", "train acc", "test acc"],
              [[p, epochs_to_fit(sweep[p]), f"{sweep[p]['train_acc'][-1]:.3f}",
                f"{sweep[p]['test_acc'][-1]:.3f}"] for p in PS])

## Summary

In all five conditions of Part 4 the network reaches approximately 100% training accuracy, while
its test accuracy ranges from chance to about 0.7. This means that none of the quantities we can
measure on the training set - the final loss, the training accuracy, the number of parameters, or
the size of the function class the architecture can represent - distinguishes the run that
generalizes from the run that has memorized noise, because all of the runs look the same from the
training side.

Taken together, these results indicate that the capacity of an architecture and the value of the
training loss are not enough to explain why a model generalizes. An explanation also has to refer
to the data the model was trained on and to the optimization procedure that produced it.